<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/IIRSI/Project/%D0%97%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B5_%D0%B4%D0%BB%D1%8F_%D1%81%D1%82%D1%83%D0%B4%D0%B5%D0%BD%D1%82%D0%BE%D0%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 **Задание для студентов: Разработка интеллектуальной системы с мультиагентной архитектурой и RAG**

---

## 🎯 **Цель проекта**

Создать **интеллектуальную систему** на любую тему (на выбор студента): машинный перевод, генерация сказок, QA-система, анализ резюме, рекомендации, суммаризация, чат-бот, проверка кода и т.д. Главное – применить современный стек технологий и архитектурные паттерны.

**Система должна включать:**
- Мультиагентную архитектуру (минимум 3 агента + Supervisor)
- Retrieval-Augmented Generation (RAG) с гибридным поиском (BM25 + векторный поиск)
- Два варианта LLM: Ollama и HuggingFace Transformers
- REST API (FastAPI)
- Асинхронную обработку, кэширование, фоновые задачи (Celery)
- Human-in-the-Loop (обратная связь и обработка неоднозначных результатов)
- Контейнеризацию (Docker)
- Тестирование (pytest)
- Простой UI (Streamlit или аналог)

**Важно:** студенты **не** используют готовые фреймворки для агентов (LangChain, LangGraph и т.п.). Вся логика агентов и оркестрации реализуется самостоятельно. Готовый код не предоставляется, даётся только описание архитектуры и требований.

---

## 🏗️ **Примерная архитектура проекта (ориентир)**

Ниже приведена структура, которую можно взять за основу. Студенты вольны менять названия файлов и папок, но обязаны реализовать все перечисленные компоненты.

```
project/
├── backend/
│   ├── agents/            # Агенты и супервизор
│   │   ├── base.py        # Абстрактный класс Agent
│   │   ├── validator.py   # Агент проверки (rule-based + LLM)
│   │   ├── critic.py      # Агент оценки качества (LLM-as-Judge)
│   │   ├── editor.py      # Агент исправления/улучшения
│   │   └── supervisor.py  # Оркестратор цикла
│   ├── api/
│   │   ├── routes/        # Роуты FastAPI
│   │   ├── deps.py        # Зависимости (auth, DB)
│   │   └── main.py        # Точка входа приложения
│   ├── core/
│   │   ├── config.py      # Настройки (pydantic-settings)
│   │   ├── constants.py   # Константы
│   │   ├── models.py      # Pydantic-модели
│   │   └── exceptions.py  # Свои исключения
│   ├── data/
│   │   ├── knowledge_base.py    # Работа с базой знаний / справочниками
│   │   ├── examples_store.py    # Работа с коллекцией примеров
│   │   ├── cache.py       # Кэш
│   │   ├── embeddings.py  # Эмбеддинги (SentenceTransformer)
│   │   ├── llm_manager.py # Выбор между Ollama и HuggingFace
│   │   └── hf_model.py    # Обёртка для HF pipeline
│   ├── db/
│   │   ├── models.py      # SQLAlchemy-модели таблиц
│   │   ├── session.py     # Подключение к БД
│   │   └── database.py    # Экспорт
│   ├── hitl/              # Human-in-the-Loop
│   │   ├── active_learning.py # Очередь неоднозначных примеров
│   │   ├── feedback.py    # Обратная связь
│   │   └── updater.py     # Обновление базы знаний
│   ├── retrieval/
│   │   ├── bm25_index.py  # BM25
│   │   ├── vector_index.py# Векторный индекс (Qdrant)
│   │   ├── hybrid_retriever.py # Гибридный поиск + RRF
│   │   └── reranker.py    # (опционально) переранжирование
│   ├── utils/
│   │   ├── llm_client.py  # Обёртка для LLM
│   │   ├── llm_processing.py # Очистка ответов
│   │   ├── async_utils.py # Retry-декоратор
│   │   ├── logging.py     # Настройка логирования
│   │   ├── security.py    # JWT, пароли
│   │   └── text_processing.py # Токенизация, нормализация
│   └── worker/            # Celery
│       ├── celery_app.py  # Приложение Celery
│       └── tasks.py       # Задачи
├── frontend/              # ИЛИ streamlit_app/
│   ├── app.py             # Главная страница
│   ├── pages/             # Дополнительные страницы
│   └── utils/
│       ├── api_client.py  # Клиент для API
│       └── constants.py   # Справочники
├── scripts/
│   ├── create_admin.py    # Создание админа
│   ├── import_data.py     # Массовый импорт CSV/JSON
│   ├── run_eval.py        # Скрипт оценки качества
│   └── import/            # Скрипты добавления данных
├── tests/
│   ├── unit/              # Модульные тесты
│   └── integration/       # Интеграционные тесты
├── infrastructure/
│   ├── docker-compose.yml # Все сервисы
│   ├── Dockerfile         # Образ приложения
│   └── nginx.conf         # Прокси
├── data/
│   └── import/            # CSV/JSON-файлы с эталонными данными
├── pyproject.toml         # Poetry
└── run.py                 # Точка входа
```

---

## 🤖 **Требования к мультиагентной системе**

1. **Базовый класс агента**
   - Абстрактный класс `Agent` с методом `async def process(state) -> state`.
   - Хранит имя агента и последний результат.
   - Все агенты наследуются от него.

2. **Валидатор (Validator)**
   - Выполняет проверку результата по формальным правилам (минимальная длина, наличие обязательных полей, отсутствие повторов и т.п.).
   - Может вызывать LLM для дополнительной проверки.
   - Возвращает список ошибок (`errors`) и предупреждений (`warnings`), а также флаг `passed`.

3. **Критик (Critic)**
   - LLM-as-a-Judge: оценивает результат по нескольким критериям (например, релевантность, полнота, стиль, точность).
   - Использует few-shot prompting с примерами хороших и плохих ответов.
   - Возвращает баллы (0–10) по каждому критерию, общий балл и рекомендации.
   - В случае ошибки LLM (таймаут, неверный JSON) возвращает fallback-оценку (например, 5.0).

4. **Редактор (Editor)**
   - Принимает ошибки/предупреждения от Validator и рекомендации от Critic.
   - Исправляет или улучшает результат через LLM.
   - Возвращает исправленный результат и флаг, изменилось ли что-то.

5. **Супервизор (Supervisor)**
   - Оркеструет работу агентов в цикле:
     ```
     Validator -> Critic -> (если нужно) Editor -> Validator -> Critic -> ...
     ```
   - Ограничивает количество итераций (`max_iterations`).
   - Определяет критерий успеха (например, `validation_passed == True` и `critic_score >= 7`).
   - Ведёт лог всех шагов (`reasoning_log`).

---

## 🔍 **Требования к RAG**

1. **Гибридный поиск**
   - Реализовать два индекса: BM25 (лексический) и векторный (семантический, Qdrant).
   - Объединить результаты методом Reciprocal Rank Fusion (RRF).

2. **Эмбеддинги**
   - Использовать `sentence-transformers` (например, `paraphrase-multilingual-MiniLM-L12-v2`).
   - Оборачивать в класс-синглтон с ленивой загрузкой.

3. **Векторная БД**
   - Qdrant (или аналог: ChromaDB, FAISS) с коллекцией и cosine-метрикой.
   - Добавление, поиск с фильтрами по категориям/языкам (если применимо).

4. **Интеграция с LLM**
   - Контекст из найденных записей подаётся в промпт генеративной модели.

---

## 🧠 **Поддержка LLM: Ollama и HuggingFace**

1. **Ollama** — основной вариант для локального запуска.
   - HTTP API (`/api/generate`).
   - Поддержка разных моделей (например, qwen2.5, mistral, llama3.1).
   - Настройка `temperature` для разных агентов.

2. **HuggingFace Transformers** — запасной вариант для случаев, когда нужна конкретная модель или когда Ollama недоступен.
   - Асинхронная обёртка над `transformers.pipeline`.
   - Кэширование результатов.

3. **Выбор бэкенда** (через `LLMManager` или аналогичный класс)
   - Если задача поддерживается Ollama — использовать его.
   - Иначе переключаться на HuggingFace.

---

## 📊 **Сбор эталонных данных (500–1000+ записей)**

Студенты должны подготовить **обучающий набор** для базы знаний и справочников (аналог глоссария).

- **Справочник / база терминов**: не менее **500** уникальных записей (пары или структурированные объекты).
- **База знаний / коллекция примеров**: не менее **1000** записей (пары "вход-выход", "вопрос-ответ", "текст-резюме" и т.п.).
- Данные можно взять из открытых источников (например, датасеты с Kaggle, OPUS, Tatoeba, готовые корпуса) или сгенерировать/составить вручную.
- Формат CSV или JSON. Колонки зависят от предметной области, но должны быть согласованы с кодом импорта.

**Скрипт генерации**:
- Читает исходные файлы (JSON, CSV или txt).
- Сэмплирует нужное количество записей.
- При необходимости подсчитывает частотность и отбирает самые частотные для справочника.
- Сохраняет результат в CSV/JSON.

**Скрипт импорта** в БД:
- Читает CSV/JSON.
- Вставляет записи в таблицы через SQLAlchemy, избегая дубликатов.

---

## 🛠️ **Backend требования**

1. **FastAPI** со следующими группами эндпоинтов (названия адаптируются под задачу):
   - Основной запрос (например, `/generate`, `/answer`, `/analyze`).
   - Мульти-запрос (несколько вариантов/языков/категорий).
   - Пакетная обработка.
   - Ансамбль моделей с выбором лучшего варианта.
   - Административные эндпоинты для управления справочником и базой знаний.
   - Аутентификация: регистрация, вход, получение информации.
   - Оценка качества (BLEU, chrF, BERTScore, METEOR для текстовых задач или свои метрики).
   - Human-in-the-Loop: получение неоднозначных примеров, подтверждение/отклонение.
   - Проверка состояния сервисов (`/health`).

2. **Аутентификация**:
   - JWT токены.
   - Роли: `user`, `editor`, `admin`.
   - Хэширование паролей (bcrypt).

3. **База данных**:
   - SQLAlchemy (async) с поддержкой PostgreSQL (основной) и SQLite (для тестов).
   - Таблицы: пользователи, справочник, база знаний, неоднозначные примеры, кэш.

4. **Кэширование**:
   - Таблица в БД или Redis.
   - Ключ: например, `(input_text, category, parameters)`.
   - Увеличивать счётчик попаданий.

5. **Фоновые задачи (Celery)**:
   - Переиндексация базы знаний.
   - Пакетная обработка.
   - Очистка устаревшего кэша.
   - Массовое добавление записей.

6. **Логирование**:
   - Структурированные логи (JSON optional).
   - Ротация файлов.
   - Разные уровни для разных модулей.

---

## 🧪 **Тестирование**

- Использовать **pytest** и **pytest-asyncio**.
- Покрытие тестами не менее **70%** (можно проверить `pytest-cov`).
- Обязательные категории:
  - Юнит-тесты для каждого агента (проверка логики, обработка ошибок).
  - Юнит-тесты для retrieval (BM25, vector, RRF).
  - Юнит-тесты для API-роутов (с mock-объектами).
  - Интеграционные тесты (через `TestClient`).
- Использовать `monkeypatch` и `unittest.mock` для изоляции внешних сервисов (Ollama, Qdrant, Redis).
- Создавать фикстуры для тестовой БД и тестовых данных.

---

## 🐳 **Инфраструктура**

1. **Poetry** для управления зависимостями (`pyproject.toml`).
2. **Docker Compose** со следующими сервисами:
   - `db` — PostgreSQL 16
   - `redis` — Redis 7
   - `qdrant` — векторная БД
   - `ollama` — LLM (с инициализацией и подгрузкой моделей)
   - `app` — FastAPI
   - `worker` — Celery worker
   - `beat` — Celery beat
   - `nginx` — прокси
3. **Healthcheck** для каждого сервиса.
4. **Nginx** конфигурация с проксированием, gzip, security headers.
5. **Dockerfile** для сборки образа приложения.

---

## 🖥️ **Frontend (Streamlit или аналог)**

- Страница входа/регистрации.
- Основная страница с формой для выполнения задачи (генерация/ответ/анализ).
- Страница пакетной обработки (загрузка файла, массовая обработка).
- Страница управления справочником и базой знаний (CRUD, импорт CSV).
- Страница Human-in-the-Loop (просмотр неоднозначных примеров, подтверждение/отклонение).
- Страница оценки качества (запуск метрик).
- Страница мониторинга (статусы сервисов, статистика кэша).

---

## 📋 **Критерии оценки**

| Критерий | Вес |
|----------|-----|
| Архитектура и чистота кода | 30% |
| Функциональность (работающие агенты, RAG, API) | 30% |
| Тестирование (покрытие, качество) | 20% |
| Документация (README, инструкции) | 10% |
| Инфраструктура (Docker, Poetry) | 10% |

**Бонусы** (до +10%): мониторинг, CI/CD, WebSocket, rate limiting, A/B тесты.

---

## 📦 **Итоговый результат**

Студент должен предоставить:

1. Репозиторий с полным кодом.
2. README с инструкциями по запуску (через Docker одной командой).
3. Набор эталонных данных (CSV/JSON) объёмом от 500 до 1000+ записей.
4. Скрипты для генерации и импорта данных.
5. Скрипт `run_eval.py` для оценки качества и сравнения режимов (baseline, +справочник, +RAG, +agents).
6. Отчёт с описанием архитектуры и принятых решений.

---

**Удачи!** Помните: главное — не просто скопировать идею, а понять принципы и реализовать свою систему.


## Этап 1. Выбор темы и настройка окружения

**Цель:** определить предметную область, установить инструменты, создать репозиторий и внешние сервисы.

**Недель:** 1.

**Задачи:**
- Выбрать тему, записать формулировку (что делает, для кого, что на входе/выходе).
- Распределить роли в команде (backend / ML-RAG / инфра-QA).
- Установить Python 3.12 (совпадает с `Dockerfile`).
- Установить VS Code + расширения: Python, Pylance, Even Better TOML, GitLens, Docker.
- Установить Git, настроить пользователя, сгенерировать SSH-ключ, добавить на GitHub.
- Зарегистрироваться на GitHub и HuggingFace, получить HF-токен типа Read.
- Установить Poetry через `pip`, использовать `python -m poetry` (надёжнее PATH).
- Включить `virtualenvs.in-project = true`.
- Установить Ollama, скачать `qwen2.5:7b`.
- Создать структуру папок (`backend/{core,api,db,agents,data,retrieval,hitl,utils,worker}`, `tests/{unit,integration}`, `infrastructure`, `scripts`, `data/import`).
- Создать `.gitignore`, `.env.example`, `.env`, `README.md`.
- Инициализировать Poetry, добавить все зависимости (список — в методичке).
- Создать репозиторий на GitHub, первый коммит.

**Результат:** репозиторий с `pyproject.toml`, `poetry.lock`, `.gitignore`, `.env.example`, README; установленные Poetry и Ollama.

**Критерии готовности:**
- [ ] Тема и роли записаны в README.
- [ ] `python --version` → `3.12.x`.
- [ ] `python -m poetry install` проходит без ошибок.
- [ ] `python -m poetry run python -c "import fastapi, sqlalchemy, celery, qdrant_client, torch; print('OK')"` выводит `OK`.
- [ ] `ollama list` показывает `qwen2.5:7b`.
- [ ] `git check-ignore -v .env` возвращает строку из `.gitignore`.
- [ ] Репозиторий запушен, в нём нет `.env`.

**Методичка:** подробная (уже написана).

---

## Этап 2. Каркас приложения

**Цель:** приложение запускается, отвечает на `/healthz`, конфиг читается из `.env`, логи пишутся в файл.

**Недель:** 1.

**Задачи:**
- Создать пустые `__init__.py` во всех пакетах.
- Реализовать `backend/core/config.py` (Pydantic Settings):
  - поля для БД, LLM (Ollama + HF), Qdrant, Redis, JWT, режимов работы;
  - properties `ASYNC_DATABASE_URL`, `ollama_url`, `qdrant_url`, `ensemble_models_list`;
  - валидатор `TRANSLATION_ENGINE in {auto, ollama, huggingface}`;
  - `model_config = {"env_file": ".env", "case_sensitive": True, "extra": "ignore"}`;
  - `@lru_cache() def get_settings()`.
- Реализовать `backend/core/constants.py` — словарь языков / категорий.
- Реализовать `backend/core/exceptions.py` — `RUNGException` и наследники.
- Реализовать `backend/utils/logging.py`:
  - `CustomFormatter` с цветным выводом;
  - `dictConfig` с хендлерами `console`, `RotatingFileHandler` (10 МБ × 10), `error_file`;
  - приглушение логов `sqlalchemy.engine`, `httpx`, `transformers` до WARNING.
- Реализовать `backend/api/main.py`: FastAPI с `lifespan`, CORS, роуты `/`, `/healthz`, `/languages`.
- Реализовать `backend/api/routes/health.py`: `/healthz` → `{"status": "ok"}`. Полноценный `/health` — на следующем этапе.
- Создать `run.py` — запуск `uvicorn backend.api.main:app --reload`.
- Написать `tests/unit/test_config.py` (чтение `.env`) и `tests/integration/test_health.py` (200 на `/healthz`).

**Результат:** `python -m poetry run uvicorn backend.api.main:app --reload` запускает приложение, `/docs` открывается, логи пишутся в `logs/`.

**Критерии готовности:**
- [ ] Структура папок совпадает с README.
- [ ] Изменение `PROJECT_NAME` в `.env` меняет заголовок в `/docs`.
- [ ] Логи есть в консоли и в `logs/rung_YYYYMMDD.log`.
- [ ] `/healthz` возвращает 200.
- [ ] Тесты проходят.

**Методичка:** краткая (полный `config.py` как шаблон, структура `logging.py`).

---

## Этап 3. База данных и минимальный Docker

**Цель:** подключить БД, описать модели, поднять Postgres и app в Docker.

**Недель:** 1.

**Задачи:**
- Реализовать `backend/db/session.py`:
  - `async_engine` с `pool_size=10`, `max_overflow=20`, `pool_pre_ping=True`;
  - `AsyncSessionLocal = async_sessionmaker(...)`;
  - `get_async_session()` для `Depends`.
- Реализовать `backend/db/models.py`:
  - `User` — id (UUID), email, username, hashed_password, full_name, role, is_active, created_at, updated_at;
  - `TMDB` — translation memory, `UniqueConstraint(source_lang, target_lang, source_text, target_text)`;
  - `GlossaryDB` — глоссарий, те же поля и constraint;
  - `UncertainExample` — очередь HITL (source, machine_translation, corrected_translation, critic_score, validation_passed, status, created_at, confirmed_at);
  - `TranslationCache` — кэш (source, translation, model_used, engine_used, hits, created_at), `UniqueConstraint(source_text, source_lang, target_lang, engine_used)`.
- Обновить `backend/api/main.py`: в `lifespan` — `Base.metadata.create_all`.
- Создать `infrastructure/Dockerfile` (минимальный):
  ```dockerfile
  FROM python:3.12-slim
  WORKDIR /app
  RUN pip install --no-cache-dir poetry && poetry config virtualenvs.create false
  COPY pyproject.toml poetry.lock ./
  RUN poetry install --no-root --only main --no-interaction
  COPY backend/ ./backend/
  COPY run.py ./
  ```
- Создать `infrastructure/docker-compose.yml` с `db` (Postgres 16) и `app`. Healthchecks оба. `app` зависит от `db` через `condition: service_healthy`.
- Обновить `.env` для Docker: `DATABASE_URL=postgresql://rung_user:rung_password@db:5432/rung_db`.
- Обновить `backend/api/routes/health.py`: `/health` проверяет БД через `SELECT 1`.
- Написать `tests/unit/test_models.py` (SQLite in-memory).

**Результат:** `docker compose -f infrastructure/docker-compose.yml up` поднимает `db` и `app`, `/health` возвращает `database: healthy`.

**Критерии готовности:**
- [ ] `Base.metadata.create_all` создаёт все таблицы при старте.
- [ ] `/health` проверяет БД.
- [ ] `docker compose ps` — оба сервиса `healthy`.
- [ ] При перезапуске данные сохраняются (volume).
- [ ] Тесты на модели проходят.

**Методичка:** подробная (SQLAlchemy async, UUID, ловушка «Future attached to a different loop», когда переходить на Alembic).

**Почему Docker уже здесь:** если отложить — 14 недель разработки на SQLite, потом неделя боли при переезде. `localhost` в контейнере ≠ `localhost` хоста, `postgresql://` ≠ `postgresql+asyncpg://`, healthcheck'и и порядок старта. Проще решить сейчас.

---

## Этап 4. Аутентификация и авторизация

**Цель:** регистрация, вход, роли через JWT.

**Недель:** 1.

**Задачи:**
- Реализовать `backend/utils/security.py`:
  - `pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")`;
  - `get_password_hash`, `verify_password`;
  - `create_access_token(data)` с `exp`;
  - `decode_token(token)` — `HTTPException(401)` при ошибке.
- Реализовать `backend/api/deps.py`:
  - `HTTPBearer(auto_error=False)`;
  - `get_db()` — yield из `AsyncSessionLocal`;
  - `get_current_user` — 401 без токена, 401 при неверном токене, 404 если юзер не найден;
  - `require_admin`, `require_editor` — 403 при несоответствии;
  - `optional_user` — не падает, возвращает `None`.
- Реализовать `backend/api/routes/auth.py`:
  - `POST /auth/register` — валидация пароля (≥8, цифра, заглавная), уникальность email/username, возврат токена;
  - `POST /auth/login` — проверка, 401 при ошибке, 403 при `is_active=False`;
  - `GET /auth/me` — текущий пользователь.
- Pydantic-модели: `RegisterRequest`, `LoginRequest`, `TokenResponse`.
- Тесты: `tests/unit/test_security.py`, `tests/integration/test_auth.py` (SQLite in-memory, полный цикл).

**Результат:** регистрация → логин → токен → `/auth/me` работает.

**Критерии готовности:**
- [ ] Пароли только в bcrypt-хеше.
- [ ] `/auth/me` без токена — 401, с токеном — данные.
- [ ] Токен живёт `JWT_EXPIRATION_MINUTES`.
- [ ] Тесты проходят без реальной Postgres.

**Методичка:** средняя (`Depends`, ловушка `passlib` + `bcrypt>=5.0`, генерация `JWT_SECRET_KEY`).

---

## Этап 5. Мультиагентная система

**Цель:** Supervisor + Validator + Critic + Editor. Сначала на заглушках LLM.

**Недель:** 2. Это самый большой этап по количеству кода.

### 5.1. Agent + Supervisor (первая неделя)

- Реализовать `backend/agents/base.py`:
  - `Agent(ABC)` с `name`, `_last_result`, `settings`, `engine`;
  - `@abstractmethod async def process(state)`;
  - `get_last_result()`.
- Реализовать `backend/agents/supervisor.py`:
  - ленивая инициализация `_get_validator`, `_get_critic`, `_get_editor`;
  - цикл: `Validator → Critic → (если нужно) Editor → Validator → Critic → ...`;
  - **критерий выхода (явно!):** `needs_correction = (not validation_passed) or (critic_score is not None and critic_score < 7.0)`. Если `critic_score is None` — только по `validation_passed`;
  - `max_iterations` = 3 по умолчанию;
  - в `state` пишутся `supervisor_iterations` и `supervisor_final`;
  - ведение `reasoning_log`.
- Тесты `tests/unit/test_supervisor.py` с mock-агентами:
  - ранний выход при `validation_passed=True` и `critic_score=8.0`;
  - полный цикл при `validation_passed=False`;
  - `reasoning_log` содержит записи всех агентов + финальную;
  - `max_iterations` не превышается.

### 5.2. Validator, Critic, Editor (вторая неделя)

- `backend/agents/validator.py` — правила: мин. длина, повторы подряд, обязательные термины из глоссария, остатки исходного текста в переводе. LLM **не использовать**.
  - В `state`: `validator_errors`, `validator_warnings`, `validation_passed`.
- `backend/agents/critic.py`:
  - `_call_llm` — заглушка, возвращающая JSON с оценками 5.0 (реальная LLM — этап 8);
  - `_build_prompt` — few-shot для вашей темы, требование «только JSON»;
  - `_parse_json_response` — три попытки (markdown-блок, `{...}`, `re.findall`);
  - `_fallback_extract` — вытягивание чисел по regexp;
  - `_get_fallback_evaluation(reason)` — 5.0 + пояснение.
  - В `state`: `critic_score`, `critic_adequacy`, `critic_fluency`, `critic_terminology`, `critic_reasoning`, `critic_suggestions`, `critic_missing_terms`, `critic_passed`.
- `backend/agents/editor.py`:
  - `_call_llm` — заглушка;
  - если ошибок/предупреждений/подсказок нет — LLM не вызывается, `editor_changed=False`;
  - если есть — вызов LLM + `clean_translation`;
  - `clean_translation(raw)` — вырезает `Translation:`, `Here is...`, markdown, префиксы.
- Тесты: `test_validator.py`, `test_critic.py`, `test_editor.py` — каждый покрывает свою логику.

**Результат:** Supervisor гоняет агентов по кругу, каждый агент работает с заглушками, тесты покрывают: ранний выход, полный цикл, парсинг JSON, fallback, условный вызов Editor.

**Критерии готовности:**
- [ ] Agent — абстрактный, `process` обязателен.
- [ ] Критерий выхода Supervisor явно реализован и протестирован.
- [ ] Critic парсит JSON в любом формате, fallback на 5.0.
- [ ] Editor не вызывает LLM без проблем.
- [ ] Тесты покрывают ≥ 15 сценариев на все четыре агента.

**Методичка:** средняя (с ABC и mock-агентами; примеры промптов для Critic и Editor).

---

## Этап 6. Сбор и импорт данных

**Цель:** датасет в БД.

**Недель:** 1.

**Задачи:**
- Подготовить данные. Требования:
  - **база знаний (TM): ≥ 300 пар**;
  - **глоссарий: ≥ 150 записей**;
  - бонус +5% за 1000+ пар TM и 500+ глоссария.
- Источники: OPUS, Tatoeba, HuggingFace Datasets, либо синтетика через LLM (с пометкой).
- Написать `scripts/generate_data.py`:
  - читает исходники (JSON / CSV / txt);
  - нормализует: убирает дубли, пустые, слишком короткие/длинные;
  - сэмплирует;
  - сохраняет `data/import/tm.csv` и `data/import/glossary.csv` с колонками `source_lang,target_lang,en,tgt`.
- Написать `scripts/import_data.py`:
  - `INSERT ... ON CONFLICT DO NOTHING` (asyncpg) — без SELECT-then-INSERT;
  - логирует `added` / `skipped`.
- Загрузить в БД, проверить через SQL:
  ```sql
  SELECT source_lang, target_lang, COUNT(*) FROM translation_memories GROUP BY 1, 2;
  SELECT source_lang, target_lang, COUNT(*) FROM glossaries GROUP BY 1, 2;
  ```
- Создать `data/import/README.md` с источниками и лицензиями.

**Результат:** в БД ≥ 300 пар TM и ≥ 150 терминов; скрипты воспроизводимы, двойной запуск не создаёт дубли.

**Критерии готовности:**
- [ ] `generate_data.py` воспроизводит CSV из исходников.
- [ ] `import_data.py` идемпотентен.
- [ ] `data/import/README.md` с источниками.
- [ ] Объёмы соответствуют требованиям.

**Методичка:** краткая (форматы CSV, `ON CONFLICT`, где брать данные).

---

## Этап 7. RAG: эмбеддинги, BM25, Qdrant, RRF

**Цель:** гибридный поиск по базе знаний.

**Недель:** 2.

### 7.1. Эмбеддинги + BM25 (первая неделя)

- `backend/data/embeddings.py`:
  - `EmbeddingModel` — синглтон через класс-переменные `_model`, `_dimension`;
  - `_load()` — ленивая загрузка `SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")`;
  - `dimension` (property), `encode(texts, normalize=True)`;
  - в `encode` — `show_progress_bar=False` без TTY.
- `backend/retrieval/bm25_index.py`:
  - `BM25Index.initialize(entries)`, `search(query, top_n)`, `add_entry(entry)`;
  - внутри — `rank_bm25.BM25Okapi`;
  - `search` возвращает `TMEntry` с `score`.
- Тесты `tests/unit/test_bm25.py`: поиск на 5 записях, пустой индекс, score > 0 только для релевантных.

### 7.2. Qdrant + RRF (вторая неделя)

- `backend/retrieval/vector_index.py`:
  - `VectorIndex` с ленивым `QdrantClient`;
  - `_ensure_connection_and_collection()` — до 5 попыток с задержкой 2 сек;
  - `_ensure_collection()` — создать с `VectorParams(size=dimension, distance=Distance.COSINE)`;
  - `add_entries(entries)` — `PointStruct` с `uuid4` id и payload;
  - `search(query, source_lang, target_lang, top_n)` — `Filter` по языкам, `query_points`;
  - при недоступном Qdrant — `[]`, без исключений.
- `backend/retrieval/hybrid_retriever.py`:
  - `reciprocal_rank_fusion(bm25, vector, k=60)` — `1/(k+rank+1)` суммируется по ключу `(en, tgt, source_lang, target_lang)`;
  - `hybrid_search(query, src, tgt, top_n)` — параллельно, RRF, топ-N.
- Обновить `docker-compose.yml`: добавить `qdrant` с volume и healthcheck.
- Тесты: `test_rrf.py` (синтетика), `test_hybrid_search.py` (mock Qdrant).

**Результат:** гибридный поиск возвращает объединённый список, BM25 и векторный индекс работают, если Qdrant недоступен — деградация до BM25.

**Критерии готовности:**
- [ ] Модель эмбеддингов загружается один раз.
- [ ] BM25 работает.
- [ ] Коллекция Qdrant создаётся автоматически.
- [ ] RRF протестирован.
- [ ] Qdrant off → только BM25, без падения.

**Методичка:** подробная (Qdrant API, cosine, RRF-формула, mock Qdrant).

---

## Этап 8. Интеграция LLM (Ollama + HuggingFace)

**Цель:** реальные LLM вместо заглушек; fallback между движками.

**Недель:** 1.

**Задачи:**
- `backend/utils/llm_client.py`:
  - `call_ollama_with_model(prompt, model_name, temperature=0.3, max_retries=3)`;
  - POST на `settings.ollama_url`, timeout 300;
  - retry при `TimeoutException`, `HTTPStatusError`, `ConnectError`;
  - вызов `clean_translation`.
- `backend/data/huggingface_translator.py`:
  - `TranslatorHuggingFace` для Seq2Seq (NLLB) и Causal LM (Qwen);
  - маппинг NLLB (`eng_Latn`, `rus_Cyrl`, `tgk_Cyrl`, ...);
  - `_load_model` под `asyncio.Lock`;
  - `_translate_seq2seq` и `_translate_causal`;
  - `torch_dtype=torch.float16` для GPU, `float32` для CPU; `device_map="auto"` при наличии GPU.
- `backend/utils/hf_llm.py` — то же для генерации агентами. **Фабрика `get_hf_llm(model_name)`** с кэшем, чтобы модель не грузилась трижды (для Critic, Editor, Validator).
- `backend/data/translator_manager.py`:
  - `_is_ollama_available()` с кэшем 30 сек;
  - `get_translator_for_engine(engine, src, tgt)`:
    - `ollama` → `None` (Ollama), но если недоступна и `FALLBACK_TO_HF_ON_ERROR=True` → HF;
    - `huggingface` → всегда HF;
    - `auto` → Ollama если доступна и язык поддерживается, иначе HF.
- Обновить агентов: заменить `_call_llm`-заглушки на реальные вызовы через `call_ollama_with_model` или `HuggingFaceLLM.generate`.
- Обновить `docker-compose.yml`: добавить `ollama` (volume, healthcheck) и `ollama-init` (одноразовый, качает модели из `.env`).
- Тесты `tests/unit/test_llm_client.py` с мок-httpx: успех, таймаут, 500.

**Результат:** агенты используют реальную LLM; если Ollama недоступна — переключаются на HF.

**Критерии готовности:**
- [ ] Ollama доступна из контейнера `app` через `ollama:11434`.
- [ ] HF-модель загружается один раз (проверить по логам).
- [ ] `TRANSLATION_ENGINE=huggingface` работает без Ollama.
- [ ] `auto` использует Ollama при доступности.
- [ ] Тесты с мок-httpx проходят.

**Методичка:** подробная (NLLB lang codes, `trust_remote_code`, `torch_dtype`, `device_map`, ловушка с тройной загрузкой).

---

## Этап 9. Основное API и интеграция

**Цель:** связать агентов, RAG, LLM и кэш через REST API.

**Недель:** 2.

### 9.1. Основной роут + кэш (первая неделя)

- `backend/api/routes/translate.py`:
  - `_build_translation_prompt(text, src, tgt, glossary, tm)` — блоки Glossary и Past translations;
  - `_perform_translation(text, src, tgt, engine=None, eval_mode=None)`:
    - кэш через `cache_manager.get_cached(text, src, tgt, engine=engine)`;
    - **если `engine is None` — искать любую запись** (не только `engine_used IS NULL`), иначе в `auto` кэш никогда не сработает;
    - `mode = eval_mode or settings.EVAL_MODE`;
    - сбор глоссария и TM по режиму: `baseline` — ничего, `glossary` — только глоссарий, `tm` — глоссарий + BM25, `full` — глоссарий + hybrid_search;
    - движок через `translator_manager.get_translator_for_engine`;
    - `mode == "full"` → Supervisor;
    - сохранить в кэш с `engine=engine_used`;
  - `POST /translate` — `Depends(get_current_user)`.
- `backend/core/models.py`: `TranslateRequest`, `TranslateResponse` (с `engine_used`, `critic_score`, `reasoning_log`).
- `backend/data/cache.py`:
  - `get_cached(text, src, tgt, engine=None)` — если `engine is None`, искать без фильтра по движку;
  - `save_cache(...)` — `INSERT ... ON CONFLICT DO NOTHING`;
  - `clear_cache(source_lang, target_lang)`.

### 9.2. Пакетные роуты + админ + ансамбль (вторая неделя)

- `POST /translate/multi` — несколько `target_langs`, `asyncio.gather` **с семафором на 4**;
- `POST /translate/batch` — массив текстов, **тоже с семафором**;
- `backend/api/routes/admin.py`:
  - CRUD для глоссария и TM (`add`, `list`, `delete`);
  - `POST /admin/glossary/import-csv`, `POST /admin/tm/import-csv` — загрузка CSV, отчёт `{added, skipped, errors}`;
  - `GET /admin/cache/stats`, `DELETE /admin/cache/clear`;
  - все роуты под `require_admin`.
- `POST /translate/ensemble`:
  - список моделей, `asyncio.gather(..., return_exceptions=True)` — не падать при падении одной;
  - для каждой — прогон через Critic;
  - возвращает `variants` и `best_translation`.
- Интеграционные тесты `tests/integration/test_api.py` через `TestClient` с мок-LLM.

**Результат:** полное API, работают все режимы, ансамбль устойчив к падению отдельной модели.

**Критерии готовности:**
- [ ] `baseline` вызывает только LLM, `full` — Supervisor.
- [ ] Кэш срабатывает во второй раз при `engine=None`.
- [ ] 50 текстов через `/batch` не валят Ollama (семафор).
- [ ] CSV-импорт идемпотентен.
- [ ] Ensemble не падает, если одна модель timeout'ит.
- [ ] `reasoning_log` возвращается в ответе.

**Методичка:** средняя (семафор, `return_exceptions`, CSV, `Depends`).

---

## Этап 10. HITL и Active Learning

**Цель:** очередь неуверенных результатов, её связь с API и TM.

**Недель:** 1.

**Задачи:**
- `backend/hitl/active_learning.py`:
  - `add_uncertain_example(...)` — с проверкой на дубль в `status="pending"`;
  - `get_pending_examples(source_lang=None, target_lang=None, limit=10, offset=0)`;
  - `count_pending(...)`;
  - `confirm_correction(example_id, corrected_translation)` — сохраняет в TM **в той же транзакции**, что и смена статуса;
  - `reject_example(example_id)`;
  - `get_stats()` — `{pending, confirmed, rejected}`.
- `backend/hitl/feedback.py` — `save_correction` для прямого пути.
- **Связка с API (критично!):** в `_perform_translation` добавить:
  ```python
  if settings.ACTIVE_LEARNING_ENABLED and critic_score is not None:
      if critic_score < 7 or not validation_passed:
          await ActiveLearningManager.add_uncertain_example(...)
  ```
- `backend/api/routes/hitl.py`:
  - `POST /hitl/save-correction` (`require_editor`);
  - `GET /hitl/uncertain`;
  - `POST /hitl/uncertain/{id}/confirm`;
  - `POST /hitl/uncertain/{id}/reject`;
  - `GET /hitl/uncertain/stats`.
  - **Валидация языка:** 2–3 символа (в проекте есть `udm`).
- Тесты `tests/unit/test_active_learning.py`: добавление, повторное добавление, confirm → TM, reject.

**Результат:** после `/translate` с `critic_score < 7` появляется запись в `/hitl/uncertain`, редактор подтверждает → TM пополняется.

**Критерии готовности:**
- [ ] После низкой оценки появляется запись в очереди.
- [ ] `confirm_correction` атомарен.
- [ ] Роуты под `editor`.
- [ ] Все четыре сценария покрыты тестами.

**Методичка:** средняя (связка API↔HITL, транзакции).

---

## Этап 11. Celery, кэш и фоновые задачи

**Цель:** тяжёлые операции в фоне, автоиндексация после HITL.

**Недель:** 1.

**Задачи:**
- `backend/worker/celery_app.py`:
  - `Celery("rung", broker=REDIS_URL, backend=REDIS_URL)`;
  - `include=["backend.worker.tasks"]`;
  - `task_serializer="json"`, `timezone="UTC"`, `task_time_limit=30 * 60`, `worker_prefetch_multiplier=1`, `task_acks_late=True`.
- `backend/worker/tasks.py`:
  - `index_translation_memory` — `tm_manager.load_all_into_cache()`;
  - `batch_translate(texts, src, tgt)`;
  - `run_evaluation_async(sources, hypotheses, references)`;
  - `clear_old_cache(days=30)`;
  - `add_glossary_bulk(entries)`, `add_tm_bulk(entries)`.
- **Правильная работа с event loop:** **не** создавать loop на каждую задачу (`asyncio.new_event_loop()` ломает `async_engine`). Один persistent loop на процесс:
  ```python
  _LOOP = asyncio.new_event_loop()
  def _run_async(coro):
      return _LOOP.run_until_complete(coro)
  ```
- **Связка с HITL (критично!):** в `ActiveLearningManager.confirm_correction` после `add_entry` вызывать `index_translation_memory.delay()`. Иначе новая запись не появится в BM25 / Qdrant.
- Обновить `docker-compose.yml`: добавить `redis`, `worker` (`--concurrency=1`), `beat`.
- `beat_schedule`: раз в сутки `clear_old_cache`.
- Тесты `tests/unit/test_tasks.py` с `task_always_eager=True`.

**Результат:** Celery-воркер работает, TM-индексы обновляются после HITL, кэш чистится по расписанию.

**Критерии готовности:**
- [ ] `docker compose ps` — `worker` и `beat` в `running`.
- [ ] `index_translation_memory.delay()` виден в логах.
- [ ] После `confirm_correction` новая TM-запись находится через `/translate`.
- [ ] `clear_old_cache` запускается по расписанию.
- [ ] Тесты с `task_always_eager` проходят.

**Методичка:** средняя (`celery_app.py` как шаблон, ловушка event loop, `beat_schedule`).

---

## Этап 12. Streamlit, Docker, финальная сборка

**Цель:** UI + production-сборка + README.

**Недель:** 1 + буфер.

### 12.1. Streamlit (2–3 страницы)

- `streamlit_app/app.py`:
  - форма логина (JWT в `st.session_state`);
  - основная страница: поле ввода, выбор языков, кнопка «Перевести», результат + `critic_score`;
  - страница HITL: очередь `uncertain`, кнопки «Подтвердить» / «Отклонить».
- `streamlit_app/utils/api_client.py` — `httpx` с `Authorization: Bearer <token>`.
- `streamlit_app/requirements.txt` — `streamlit`, `httpx`.

### 12.2. Docker, Nginx, README, финальный тест

- Обновить `docker-compose.yml`: добавить `nginx` (прокси на `app:8000`, порт 80), `streamlit` (порт 8501).
- `infrastructure/nginx.conf`: `proxy_pass http://app:8000`, gzip, security headers, `proxy_read_timeout 600s`.
- Проверить полный стек: `/health`, `/docs`, `/translate`, Streamlit, Celery.
- Обновить `README.md`:
  - «Быстрый старт» — одна команда;
  - «Архитектура» — диаграмма;
  - «Данные» — источники, объёмы;
  - «Скриншоты»;
  - «Известные ограничения» (честность важнее).
- Написать `tests/integration/test_end_to_end.py`: регистрация → логин → перевод → кэш → HITL.

**Результат:** проект запускается одной командой `docker compose up -d`, всё работает.

**Критерии готовности:**
- [ ] Все сервисы `healthy` в `docker compose ps`.
- [ ] `/health` возвращает 200.
- [ ] Streamlit открывается на `:8501`.
- [ ] README: описание, стек, скриншоты, запуск, ограничения.
- [ ] `pytest -v` проходит, покрытие ≥ 60%.
- [ ] Демонстрация: перевод end-to-end, HITL, fallback на HF.

**Методичка:** подробная (Nginx-конфиг, сеть Docker, «localhost ≠ localhost», healthcheck).

---

## 📌 Общие правила

**Git:**
- Ветка `week-N` для каждого этапа, merge в `main` через PR.
- Коммит `feat(week-N): ...` в конце этапа.
- Отчёт `docs/week-N.md` — что сделано, что не получилось, что перенесено.

**Тесты:**
- Пишутся вместе с кодом.
- Покрытие ≥ 60%.
- Внешние сервисы (Ollama, Qdrant, Postgres) — mock или SQLite in-memory.
- Обязательные тесты: Supervisor (цикл), Critic (парсинг + fallback), Editor (условный вызов), RRF, HITL (атомарность), кэш (срабатывание при `engine=None`).

**Что оценивается на защите:**
- Живой прогон через pipeline с `reasoning_log`.
- Выключить Ollama → система переключается на HF.
- `critic_score < 7` → запись в HITL → подтверждение → TM пополняется → повторный перевод использует новую запись.
- Вопросы: «Почему RRF?», «Что если Qdrant недоступен?», «Где Supervisor может зациклиться?», «Почему кэш хранит `engine_used`?».

**Бонусы (до +10%):**
- Ансамбль моделей с голосованием Critic.
- Alembic-миграции вместо `create_all`.
- CI/CD через GitHub Actions (тесты на каждый PR).
- Метрики Prometheus + Grafana.
- WebSocket для прогресса длинных переводов.
- Rate limiting через `slowapi`.
